In [1]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T

# https://github.com/nipponjo/deepfillv2-pytorch/tree/master
# =========================================================
# 基础工具
# =========================================================
def _init_conv_layer(conv, activation, mode='fan_out'):
    if isinstance(activation, nn.LeakyReLU):
        torch.nn.init.kaiming_uniform_(
            conv.weight,
            a=activation.negative_slope,
            nonlinearity='leaky_relu',
            mode=mode
        )
    elif isinstance(activation, (nn.ReLU, nn.ELU)):
        torch.nn.init.kaiming_uniform_(
            conv.weight,
            nonlinearity='relu',
            mode=mode
        )
    if conv.bias is not None:
        torch.nn.init.zeros_(conv.bias)


def output_to_image(out):
    out = (out[0].cpu().permute(1, 2, 0) + 1.) * 127.5
    out = out.to(torch.uint8).numpy()
    return out


def same_padding(images, ksizes, strides, rates):
    """
    TensorFlow SAME padding
    """
    in_height, in_width = images.shape[2:]
    out_height = -(in_height // -strides[0])  # ceil
    out_width = -(in_width // -strides[1])

    filter_height = (ksizes[0] - 1) * rates[0] + 1
    filter_width = (ksizes[1] - 1) * rates[1] + 1

    pad_along_height = max((out_height - 1) * strides[0] + filter_height - in_height, 0)
    pad_along_width = max((out_width - 1) * strides[1] + filter_width - in_width, 0)

    pad_top = pad_along_height // 2
    pad_bottom = pad_along_height - pad_top
    pad_left = pad_along_width // 2
    pad_right = pad_along_width - pad_left

    paddings = (pad_left, pad_right, pad_top, pad_bottom)
    padded_images = torch.nn.ZeroPad2d(paddings)(images)
    return padded_images


def downsampling_nn_tf(images, n=2):
    """
    TensorFlow 风格最近邻下采样
    """
    in_height, in_width = images.shape[2:]
    out_height, out_width = in_height // n, in_width // n
    height_inds = torch.linspace(
        0, in_height - 1, steps=out_height, device=images.device
    ).add_(0.5).floor_().long()
    width_inds = torch.linspace(
        0, in_width - 1, steps=out_width, device=images.device
    ).add_(0.5).floor_().long()
    return images[:, :, height_inds][..., width_inds]


def extract_image_patches(images, ksizes, strides, rates, padding='same'):
    """
    Extract sliding local blocks
    """
    if padding == 'same':
        images = same_padding(images, ksizes, strides, rates)
        padding = 0

    unfold = torch.nn.Unfold(
        kernel_size=ksizes,
        stride=strides,
        padding=padding,
        dilation=rates
    )
    patches = unfold(images)
    return patches


# =========================================================
# Flow 可视化相关（contextual attention 会用到）
# =========================================================
def flow_to_image(flow):
    out = []
    maxu = -999.
    maxv = -999.
    minu = 999.
    minv = 999.
    maxrad = -1
    for i in range(flow.shape[0]):
        u = flow[i, :, :, 0]
        v = flow[i, :, :, 1]
        idxunknow = (abs(u) > 1e7) | (abs(v) > 1e7)
        u[idxunknow] = 0
        v[idxunknow] = 0
        maxu = max(maxu, np.max(u))
        minu = min(minu, np.min(u))
        maxv = max(maxv, np.max(v))
        minv = min(minv, np.min(v))
        rad = np.sqrt(u ** 2 + v ** 2)
        maxrad = max(maxrad, np.max(rad))
        u = u / (maxrad + np.finfo(float).eps)
        v = v / (maxrad + np.finfo(float).eps)
        img = compute_color(u, v)
        out.append(img)
    return np.float32(np.uint8(out))


def compute_color(u, v):
    h, w = u.shape
    img = np.zeros([h, w, 3])
    nanIdx = np.isnan(u) | np.isnan(v)
    u[nanIdx] = 0
    v[nanIdx] = 0
    colorwheel = make_color_wheel()
    ncols = np.size(colorwheel, 0)
    rad = np.sqrt(u ** 2 + v ** 2)
    a = np.arctan2(-v, -u) / np.pi
    fk = (a + 1) / 2 * (ncols - 1) + 1
    k0 = np.floor(fk).astype(int)
    k1 = k0 + 1
    k1[k1 == ncols + 1] = 1
    f = fk - k0
    for i in range(np.size(colorwheel, 1)):
        tmp = colorwheel[:, i]
        col0 = tmp[k0 - 1] / 255
        col1 = tmp[k1 - 1] / 255
        col = (1 - f) * col0 + f * col1
        idx = rad <= 1
        col[idx] = 1 - rad[idx] * (1 - col[idx])
        notidx = np.logical_not(idx)
        col[notidx] *= 0.75
        img[:, :, i] = np.uint8(np.floor(255 * col * (1 - nanIdx)))
    return img


def make_color_wheel():
    RY, YG, GC, CB, BM, MR = (15, 6, 4, 11, 13, 6)
    ncols = RY + YG + GC + CB + BM + MR
    colorwheel = np.zeros([ncols, 3])
    col = 0

    colorwheel[0:RY, 0] = 255
    colorwheel[0:RY, 1] = np.transpose(np.floor(255 * np.arange(0, RY) / RY))
    col += RY

    colorwheel[col:col + YG, 0] = 255 - np.transpose(np.floor(255 * np.arange(0, YG) / YG))
    colorwheel[col:col + YG, 1] = 255
    col += YG

    colorwheel[col:col + GC, 1] = 255
    colorwheel[col:col + GC, 2] = np.transpose(np.floor(255 * np.arange(0, GC) / GC))
    col += GC

    colorwheel[col:col + CB, 1] = 255 - np.transpose(np.floor(255 * np.arange(0, CB) / CB))
    colorwheel[col:col + CB, 2] = 255
    col += CB

    colorwheel[col:col + BM, 2] = 255
    colorwheel[col:col + BM, 0] = np.transpose(np.floor(255 * np.arange(0, BM) / BM))
    col += BM

    colorwheel[col:col + MR, 2] = 255 - np.transpose(np.floor(255 * np.arange(0, MR) / MR))
    colorwheel[col:col + MR, 0] = 255
    return colorwheel


# =========================================================
# Gated Convolution
# =========================================================
class GConv(nn.Module):
    """
    Gated Convolution (TensorFlow-style SAME padding)
    """

    def __init__(self, cnum_in, cnum_out, ksize, stride=1, rate=1,
                 padding='same', activation=nn.ELU()):
        super().__init__()

        self.activation = activation
        self.cnum_out = cnum_out
        num_conv_out = cnum_out if self.cnum_out == 3 or self.activation is None else 2 * cnum_out

        self.conv = nn.Conv2d(
            cnum_in,
            num_conv_out,
            kernel_size=ksize,
            stride=stride,
            padding=0,
            dilation=rate
        )
        _init_conv_layer(self.conv, activation=self.activation)

        self.ksize = ksize
        self.stride = stride
        self.rate = rate
        self.padding = padding

    def forward(self, x):
        x = same_padding(
            x,
            [self.ksize, self.ksize],
            [self.stride, self.stride],
            [self.rate, self.rate]
        )

        x = self.conv(x)

        if self.cnum_out == 3 or self.activation is None:
            return x

        x, y = torch.split(x, self.cnum_out, dim=1)
        x = self.activation(x)
        y = torch.sigmoid(y)
        x = x * y
        return x


class GDeConv(nn.Module):
    """
    Upsampling followed by GConv
    """
    def __init__(self, cnum_in, cnum_out, padding=1):
        super().__init__()
        self.conv = GConv(cnum_in, cnum_out, 3, 1, padding=padding)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode='nearest', recompute_scale_factor=False)
        x = self.conv(x)
        return x


# =========================================================
# Contextual Attention
# =========================================================
class ContextualAttention(nn.Module):
    def __init__(self, ksize=3, stride=1, rate=1, fuse_k=3,
                 softmax_scale=10., n_down=2, fuse=True,
                 return_flow=False, device_ids=None):
        super().__init__()
        self.ksize = ksize
        self.stride = stride
        self.rate = rate
        self.fuse_k = fuse_k
        self.softmax_scale = softmax_scale
        self.fuse = fuse
        self.device_ids = device_ids
        self.n_down = n_down
        self.return_flow = return_flow

    def forward(self, f, b, mask=None):
        device = f.device
        raw_int_fs, raw_int_bs = list(f.size()), list(b.size())

        kernel = 2 * self.rate
        raw_w = extract_image_patches(
            b,
            ksizes=[kernel, kernel],
            strides=[self.rate * self.stride, self.rate * self.stride],
            rates=[1, 1],
            padding='same'
        )
        raw_w = raw_w.view(raw_int_bs[0], raw_int_bs[1], kernel, kernel, -1)
        raw_w = raw_w.permute(0, 4, 1, 2, 3)
        raw_w_groups = torch.split(raw_w, 1, dim=0)

        f = downsampling_nn_tf(f, n=self.rate)
        b = downsampling_nn_tf(b, n=self.rate)
        int_fs, int_bs = list(f.size()), list(b.size())

        f_groups = torch.split(f, 1, dim=0)

        w = extract_image_patches(
            b,
            ksizes=[self.ksize, self.ksize],
            strides=[self.stride, self.stride],
            rates=[1, 1],
            padding='same'
        )
        w = w.view(int_bs[0], int_bs[1], self.ksize, self.ksize, -1)
        w = w.permute(0, 4, 1, 2, 3)
        w_groups = torch.split(w, 1, dim=0)

        if mask is None:
            mask = torch.zeros([int_bs[0], 1, int_bs[2], int_bs[3]], device=device)
        else:
            mask = downsampling_nn_tf(mask, n=(2 ** self.n_down) * self.rate)

        int_ms = list(mask.size())
        m = extract_image_patches(
            mask,
            ksizes=[self.ksize, self.ksize],
            strides=[self.stride, self.stride],
            rates=[1, 1],
            padding='same'
        )
        m = m.view(int_ms[0], int_ms[1], self.ksize, self.ksize, -1)
        m = m.permute(0, 4, 1, 2, 3)
        m = m[0]

        mm = (torch.mean(m, axis=[1, 2, 3], keepdim=True) == 0.).to(torch.float32)
        mm = mm.permute(1, 0, 2, 3)

        y = []
        offsets = []
        k = self.fuse_k
        scale = self.softmax_scale
        fuse_weight = torch.eye(k, device=device).view(1, 1, k, k)

        for xi, wi, raw_wi in zip(f_groups, w_groups, raw_w_groups):
            wi = wi[0]
            max_wi = torch.sqrt(torch.sum(torch.pow(wi, 2), dim=[1, 2, 3], keepdim=True)).clamp_min(1e-4)
            wi_normed = wi / max_wi

            xi = same_padding(xi, [self.ksize, self.ksize], [1, 1], [1, 1])
            yi = F.conv2d(xi, wi_normed, stride=1)

            if self.fuse:
                yi = yi.view(1, 1, int_bs[2] * int_bs[3], int_fs[2] * int_fs[3])
                yi = same_padding(yi, [k, k], [1, 1], [1, 1])
                yi = F.conv2d(yi, fuse_weight, stride=1)
                yi = yi.contiguous().view(1, int_bs[2], int_bs[3], int_fs[2], int_fs[3])
                yi = yi.permute(0, 2, 1, 4, 3).contiguous()

                yi = yi.view(1, 1, int_bs[2] * int_bs[3], int_fs[2] * int_fs[3])
                yi = same_padding(yi, [k, k], [1, 1], [1, 1])
                yi = F.conv2d(yi, fuse_weight, stride=1)
                yi = yi.contiguous().view(1, int_bs[3], int_bs[2], int_fs[3], int_fs[2])
                yi = yi.permute(0, 2, 1, 4, 3).contiguous()

            yi = yi.view(1, int_bs[2] * int_bs[3], int_fs[2], int_fs[3])

            yi = yi * mm
            yi = F.softmax(yi * scale, dim=1)
            yi = yi * mm

            if self.return_flow:
                offset = torch.argmax(yi, dim=1, keepdim=True)

                if int_bs != int_fs:
                    times = (int_fs[2] * int_fs[3]) / (int_bs[2] * int_bs[3])
                    offset = ((offset + 1) * times - 1).to(torch.int64)

                offset = torch.cat([
                    torch.div(offset, int_fs[3], rounding_mode='trunc'),
                    offset % int_fs[3]
                ], dim=1)
                offsets.append(offset)

            wi_center = raw_wi[0]
            yi = F.conv_transpose2d(yi, wi_center, stride=self.rate, padding=1) / 4.
            y.append(yi)

        y = torch.cat(y, dim=0)
        y = y.contiguous().view(raw_int_fs)

        if not self.return_flow:
            return y, None

        offsets = torch.cat(offsets, dim=0)
        offsets = offsets.view(int_fs[0], 2, *int_fs[2:])

        h_add = torch.arange(int_fs[2], device=device).view([1, 1, int_fs[2], 1]).expand(int_fs[0], -1, -1, int_fs[3])
        w_add = torch.arange(int_fs[3], device=device).view([1, 1, 1, int_fs[3]]).expand(int_fs[0], -1, int_fs[2], -1)
        offsets = offsets - torch.cat([h_add, w_add], dim=1)

        flow = torch.from_numpy(flow_to_image(offsets.permute(0, 2, 3, 1).detach().cpu().numpy())) / 255.
        flow = flow.permute(0, 3, 1, 2)

        if self.rate != 1:
            flow = F.interpolate(flow, scale_factor=self.rate, mode='bilinear', align_corners=True)

        return y, flow


# =========================================================
# DeepFill v2 Generator (networks_tf style)
# =========================================================
class Generator(nn.Module):
    def __init__(self, cnum_in=5, cnum=48, return_flow=False, checkpoint=None):
        super().__init__()

        # ---------------- Stage 1 ----------------
        self.conv1 = GConv(cnum_in, cnum // 2, 5, 1, padding=2)
        self.conv2_downsample = GConv(cnum // 2, cnum, 3, 2)
        self.conv3 = GConv(cnum, cnum, 3, 1)
        self.conv4_downsample = GConv(cnum, 2 * cnum, 3, 2)
        self.conv5 = GConv(2 * cnum, 2 * cnum, 3, 1)

        self.conv6 = GConv(2 * cnum, 2 * cnum, 3, 1)
        self.conv7_atrous = GConv(2 * cnum, 2 * cnum, 3, rate=2, padding=2)
        self.conv8_atrous = GConv(2 * cnum, 2 * cnum, 3, rate=4, padding=4)
        self.conv9_atrous = GConv(2 * cnum, 2 * cnum, 3, rate=8, padding=8)
        self.conv10_atrous = GConv(2 * cnum, 2 * cnum, 3, rate=16, padding=16)
        self.conv11 = GConv(2 * cnum, 2 * cnum, 3, 1)
        self.conv12 = GConv(2 * cnum, 2 * cnum, 3, 1)

        self.conv13_upsample = GDeConv(2 * cnum, cnum)
        self.conv14 = GConv(cnum, cnum, 3, 1)
        self.conv15_upsample = GDeConv(cnum, cnum // 2)
        self.conv16 = GConv(cnum // 2, cnum // 4, 3, 1)
        self.conv17 = GConv(cnum // 4, 3, 3, 1, activation=None)
        self.tanh = nn.Tanh()

        # ---------------- Stage 2 conv branch ----------------
        self.xconv1 = GConv(3, cnum // 2, 5, 1, padding=2)
        self.xconv2_downsample = GConv(cnum // 2, cnum // 2, 3, 2)
        self.xconv3 = GConv(cnum // 2, cnum, 3, 1)
        self.xconv4_downsample = GConv(cnum, cnum, 3, 2)
        self.xconv5 = GConv(cnum, 2 * cnum, 3, 1)

        self.xconv6 = GConv(2 * cnum, 2 * cnum, 3, 1)
        self.xconv7_atrous = GConv(2 * cnum, 2 * cnum, 3, rate=2, padding=2)
        self.xconv8_atrous = GConv(2 * cnum, 2 * cnum, 3, rate=4, padding=4)
        self.xconv9_atrous = GConv(2 * cnum, 2 * cnum, 3, rate=8, padding=8)
        self.xconv10_atrous = GConv(2 * cnum, 2 * cnum, 3, rate=16, padding=16)

        # ---------------- Stage 2 attention branch ----------------
        self.pmconv1 = GConv(3, cnum // 2, 5, 1, padding=2)
        self.pmconv2_downsample = GConv(cnum // 2, cnum // 2, 3, 2)
        self.pmconv3 = GConv(cnum // 2, cnum, 3, 1)
        self.pmconv4_downsample = GConv(cnum, 2 * cnum, 3, 2)
        self.pmconv5 = GConv(2 * cnum, 2 * cnum, 3, 1)

        self.pmconv6 = GConv(2 * cnum, 2 * cnum, 3, 1, activation=nn.ReLU())
        self.contextual_attention = ContextualAttention(
            ksize=3,
            stride=1,
            rate=2,
            fuse_k=3,
            softmax_scale=10,
            fuse=False,
            device_ids=None,
            n_down=2,
            return_flow=return_flow
        )
        self.pmconv9 = GConv(2 * cnum, 2 * cnum, 3, 1)
        self.pmconv10 = GConv(2 * cnum, 2 * cnum, 3, 1)

        self.allconv11 = GConv(4 * cnum, 2 * cnum, 3, 1)
        self.allconv12 = GConv(2 * cnum, 2 * cnum, 3, 1)
        self.allconv13_upsample = GDeConv(2 * cnum, cnum)
        self.allconv14 = GConv(cnum, cnum, 3, 1)
        self.allconv15_upsample = GDeConv(cnum, cnum // 2)
        self.allconv16 = GConv(cnum // 2, cnum // 4, 3, 1)
        self.allconv17 = GConv(cnum // 4, 3, 3, 1, activation=None)

        self.return_flow = return_flow

        if checkpoint is not None:
            generator_state_dict = torch.load(checkpoint, map_location="cpu")['G']
            self.load_state_dict(generator_state_dict, strict=True)

        self.eval()

    def forward(self, x, mask):
        xin = x

        # stage 1
        x = self.conv1(x)
        x = self.conv2_downsample(x)
        x = self.conv3(x)
        x = self.conv4_downsample(x)
        x = self.conv5(x)

        x = self.conv6(x)
        x = self.conv7_atrous(x)
        x = self.conv8_atrous(x)
        x = self.conv9_atrous(x)
        x = self.conv10_atrous(x)
        x = self.conv11(x)
        x = self.conv12(x)

        x = self.conv13_upsample(x)
        x = self.conv14(x)
        x = self.conv15_upsample(x)
        x = self.conv16(x)

        x = self.conv17(x)
        x = self.tanh(x)
        x_stage1 = x

        # stage 2
        x = x * mask + xin[:, 0:3, :, :] * (1. - mask)

        # conv branch
        xnow = x
        x = self.xconv1(xnow)
        x = self.xconv2_downsample(x)
        x = self.xconv3(x)
        x = self.xconv4_downsample(x)
        x = self.xconv5(x)

        x = self.xconv6(x)
        x = self.xconv7_atrous(x)
        x = self.xconv8_atrous(x)
        x = self.xconv9_atrous(x)
        x = self.xconv10_atrous(x)
        x_hallu = x

        # attention branch
        x = self.pmconv1(xnow)
        x = self.pmconv2_downsample(x)
        x = self.pmconv3(x)
        x = self.pmconv4_downsample(x)
        x = self.pmconv5(x)

        x = self.pmconv6(x)
        x, offset_flow = self.contextual_attention(x, x, mask)
        x = self.pmconv9(x)
        x = self.pmconv10(x)
        pm = x

        x = torch.cat([x_hallu, pm], dim=1)

        x = self.allconv11(x)
        x = self.allconv12(x)
        x = self.allconv13_upsample(x)
        x = self.allconv14(x)
        x = self.allconv15_upsample(x)
        x = self.allconv16(x)

        x = self.allconv17(x)
        x = self.tanh(x)
        x_stage2 = x

        if self.return_flow:
            return x_stage1, x_stage2, offset_flow

        return x_stage1, x_stage2

In [2]:
# =========================================================
# 加载权重
# 默认适配 states_tf_places2.pth 一类权重
# =========================================================
def load_model(weight_path, device):
    checkpoint = torch.load(weight_path, map_location=device)

    if "G" not in checkpoint:
        raise KeyError("权重文件中没有找到 'G'，请确认下载的是 DeepFill v2 生成器权重。")

    generator_state_dict = checkpoint["G"]

    model = Generator(cnum_in=5, cnum=48, return_flow=False).to(device)
    model.load_state_dict(generator_state_dict, strict=True)
    model.eval()

    print("✅ DeepFill v2 权重加载完成")
    return model


# =========================================================
# 在 resize 后的图像上生成 mask
# 对 DeepFill v2：
# mask = 1 表示待修复区域
# mask = 0 表示保留区域
# =========================================================
def create_mask_from_img(img_bgr, percentile=99.5):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    threshold = np.percentile(gray, percentile)
    _, mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)

    mask = (mask.astype(np.float32) / 255.0)
    return mask


# =========================================================
# 单张图像修复
# =========================================================
def inpaint(imgpath, model, device, percentile=99.5, target_size=(384, 256)):
    img = cv2.imread(imgpath)
    if img is None:
        raise ValueError(f"无法读取图像: {imgpath}")

    # 1. resize
    img = cv2.resize(img, target_size)

    # 2. 生成 99.5% mask
    mask = create_mask_from_img(img, percentile=percentile)

    # 3. BGR -> RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)

    # 4. 转 tensor
    image = T.ToTensor()(img_pil)  # [0,1], (3,H,W)
    mask_t = torch.from_numpy(mask).unsqueeze(0)  # (1,H,W)

    _, h, w = image.shape
    grid = 8
    h = (h // grid) * grid
    w = (w // grid) * grid

    image = image[:3, :h, :w].unsqueeze(0)      # (1,3,H,W)
    mask_t = mask_t[:, :h, :w].unsqueeze(0)     # (1,1,H,W)

    image = (image * 2.0 - 1.0).to(device)      # [-1,1]
    mask_t = (mask_t > 0.5).float().to(device)  # 1=masked, 0=known

    # 5. 组装 5 通道输入
    image_masked = image * (1.0 - mask_t)
    ones_x = torch.ones_like(image_masked)[:, 0:1, :, :]
    x = torch.cat([image_masked, ones_x, ones_x * mask_t], dim=1)

    # 6. 推理
    with torch.inference_mode():
        _, x_stage2 = model(x, mask_t)
        image_inpainted = image * (1.0 - mask_t) + x_stage2 * mask_t

    # 7. 反归一化
    img_out = ((image_inpainted[0].permute(1, 2, 0) + 1.0) * 127.5)
    img_out = img_out.clamp(0, 255).to(device="cpu", dtype=torch.uint8).numpy()

    # RGB -> BGR
    img_out = cv2.cvtColor(img_out, cv2.COLOR_RGB2BGR)
    return img_out

In [3]:
# =========================================================
# 批处理
# =========================================================
def process(input_dir, output_dir, model, device, percentile=99.5, target_size=(384, 256)):
    os.makedirs(output_dir, exist_ok=True)

    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
    files = [f for f in os.listdir(input_dir) if f.lower().endswith(valid_exts)]
    files.sort()

    for f in files:
        path = os.path.join(input_dir, f)

        try:
            result = inpaint(
                imgpath=path,
                model=model,
                device=device,
                percentile=percentile,
                target_size=target_size
            )

            stem = os.path.splitext(f)[0]
            save_path = os.path.join(output_dir, stem + ".png")
            cv2.imwrite(save_path, result)

            print(f"✔ {f}")

        except Exception as e:
            print(f"✘ 处理失败: {f} | {e}")

In [5]:
# =========================================================
# 主函数（单个文件夹版本）
# =========================================================
if __name__ == "__main__":
    # 这里填你下载好的权重
    weight_path = "./Deepfillv2-states_tf_places2.pth"

    # 这里填单个输入文件夹
    input_path = r"F:\polyp\Train PolypGen+CVC\img"

    # 这里填单个输出文件夹
    output_path = r"F:\polyp-learningbased\2.99.5%+DeepFillv2\Train PolypGen+CVC\img"

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    model = load_model(weight_path, device)

    print(f"\n==============================")
    print(f"开始处理单个文件夹")
    print(f"输入目录: {input_path}")
    print(f"输出目录: {output_path}")
    print(f"==============================")

    process(
        input_dir=input_path,
        output_dir=output_path,
        model=model,
        device=device,
        percentile=99.5,
        target_size=(384, 256)
    )

    print("\n🎉 完成")

device: cuda
✅ DeepFill v2 权重加载完成

开始处理单个文件夹
输入目录: F:\polyp\Train PolypGen+CVC\img
输出目录: F:\polyp-learningbased\2.99.5%+DeepFillv2\Train PolypGen+CVC\img
✔ 1.png
✔ 10.png
✔ 101.png
✔ 102.png
✔ 103.png
✔ 104.png
✔ 105.png
✔ 107.png
✔ 108.png
✔ 109.png
✔ 11.png
✔ 110.png
✔ 111.png
✔ 112.png
✔ 113.png
✔ 114.png
✔ 115.png
✔ 116.png
✔ 117.png
✔ 118.png
✔ 12.png
✔ 120.png
✔ 121.png
✔ 122.png
✔ 123.png
✔ 124.png
✔ 125.png
✔ 126.png
✔ 127.png
✔ 128.png
✔ 129.png
✔ 13.png
✔ 130.png
✔ 131.png
✔ 132.png
✔ 133.png
✔ 135.png
✔ 136.png
✔ 137.png
✔ 138.png
✔ 139.png
✔ 140.png
✔ 141.png
✔ 142.png
✔ 143.png
✔ 144.png
✔ 145.png
✔ 146.png
✔ 147.png
✔ 149.png
✔ 15.png
✔ 150.png
✔ 151.png
✔ 152.png
✔ 153.png
✔ 155.png
✔ 156.png
✔ 157.png
✔ 158.png
✔ 159.png
✔ 16.png
✔ 160.png
✔ 161.png
✔ 162.png
✔ 164.png
✔ 165.png
✔ 167.png
✔ 168.png
✔ 169.png
✔ 17.png
✔ 170.png
✔ 172.png
✔ 173.png
✔ 174.png
✔ 175.png
✔ 176.png
✔ 177.png
✔ 178.png
✔ 18.png
✔ 180.png
✔ 182.png
✔ 183.png
✔ 184.png
✔ 186.png
✔ 187.png
✔ 188.

In [16]:

# =========================================================
# 主函数
# =========================================================
if __name__ == "__main__":
    # 这里填你下载好的权重
    # 例如：states_tf_places2.pth
    weight_path = "./Deepfillv2-states_tf_places2.pth"

    test_sets = [
        r'F:\polyp\TestDataset\CVC-300',
        r'F:\polyp\TestDataset\CVC-ClinicDB',
        r'F:\polyp\TestDataset\CVC-ColonDB',
        r'F:\polyp\TestDataset\ETIS-LaribPolypDB',
        r'F:\polyp\TestDataset\Kvasir'
    ]

    output_root = r'F:\polyp-learningbased\2.99.5%+DeepFillv2\TestDataset'

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    model = load_model(weight_path, device)
    model.eval()

    for dataset_root in test_sets:
        dataset_name = os.path.basename(os.path.normpath(dataset_root))

        input_path = os.path.join(dataset_root, 'images')
        output_path = os.path.join(output_root, dataset_name, 'images')

        print(f"\n==== 开始处理 {dataset_name} ====")
        print(f"输入目录: {input_path}")
        print(f"输出目录: {output_path}")

        process(
            input_dir=input_path,
            output_dir=output_path,
            model=model,
            device=device,
            percentile=99.5,
            target_size=(384, 256)
        )

    print("🎉 全部完成")

device: cuda
✅ DeepFill v2 权重加载完成

==== 开始处理 CVC-300 ====
输入目录: F:\polyp\TestDataset\CVC-300\images
输出目录: F:\polyp-learningbased\2.99.5%+DeepFillv2\TestDataset\CVC-300\images
✔ 149.png
✔ 150.png
✔ 151.png
✔ 152.png
✔ 153.png
✔ 154.png
✔ 155.png
✔ 156.png
✔ 157.png
✔ 158.png
✔ 159.png
✔ 160.png
✔ 161.png
✔ 162.png
✔ 163.png
✔ 164.png
✔ 165.png
✔ 166.png
✔ 167.png
✔ 168.png
✔ 169.png
✔ 170.png
✔ 171.png
✔ 172.png
✔ 173.png
✔ 174.png
✔ 175.png
✔ 176.png
✔ 177.png
✔ 178.png
✔ 179.png
✔ 180.png
✔ 181.png
✔ 182.png
✔ 183.png
✔ 184.png
✔ 185.png
✔ 186.png
✔ 187.png
✔ 188.png
✔ 189.png
✔ 190.png
✔ 191.png
✔ 192.png
✔ 193.png
✔ 194.png
✔ 195.png
✔ 196.png
✔ 197.png
✔ 198.png
✔ 199.png
✔ 200.png
✔ 201.png
✔ 202.png
✔ 203.png
✔ 204.png
✔ 205.png
✔ 206.png
✔ 207.png
✔ 208.png

==== 开始处理 CVC-ClinicDB ====
输入目录: F:\polyp\TestDataset\CVC-ClinicDB\images
输出目录: F:\polyp-learningbased\2.99.5%+DeepFillv2\TestDataset\CVC-ClinicDB\images
✔ 100.png
✔ 106.png
✔ 119.png
✔ 134.png
✔ 14.png
✔ 148.png
✔ 154.png
